In [1]:
import pandas as pd

def cargar_feed(ruta):
    try:
        df = pd.read_csv(ruta)
        return df
    except FileNotFoundError:
        print(f"No se encontró el archivo: {ruta}")
        return None


resultado = cargar_feed("feed_corrupto.csv")
print(resultado)

print("Hola mundo")



    pedido_id  restaurante_id  categoria          city   precio  calificacion
0     PED1000              39  Ensaladas      Medellín  57430.0           4.0
1     PED1001              29      Sushi      Medellín  44275.0           3.8
2     PED1002              15    Bebidas          Cali  53654.0           4.6
3     PED1003              43      Pasta  Barranquilla  46599.0           4.3
4     PED1004               8    Burgers      Medellín  $40.682           4.6
..        ...             ...        ...           ...      ...           ...
195   PED1195              37      Sushi  Barranquilla  54534.0           3.7
196   PED1196              33      Pizza  Barranquilla  63175.0           4.5
197   PED1197              42      Pasta          Cali  13039.0          10.0
198   PED1198              44      Pizza      Medellín  $53.134           3.3
199   PED1199              24  Ensaladas        Bogotá  41627.0           NaN

[200 rows x 6 columns]
Hola mundo


In [3]:
ESQUEMA_ESPERADO = {"pedido_id", "restaurante_id", "categoria",
                    "ciudad", "precio", "calificacion"}

def validar_feed(df):
    errores = []

    if set(df.columns) != ESQUEMA_ESPERADO:
        faltan = ESQUEMA_ESPERADO - set(df.columns)
        sobran = set(df.columns) - ESQUEMA_ESPERADO
        errores.append(f"Esquema cambió. Faltan: {faltan} | Sobran: {sobran}")
        return errores

    if pd.to_numeric(df["precio"], errors="coerce").isna().any():
        cuantos = int(pd.to_numeric(df["precio"], errors="coerce").isna().sum())
        errores.append(f"{cuantos} precios no son numéricos")

    nulos = int(df["calificacion"].isna().sum())
    if nulos > 0:
        errores.append(f"{nulos} nulos en calificacion")

    cal = pd.to_numeric(df["calificacion"], errors="coerce")
    fuera = int(((cal < 3) | (cal > 5)).sum())
    if fuera > 0:
        errores.append(f"{fuera} calificaciones fuera de rango [3-5]")

    return errores

In [4]:
df_sano = pd.read_csv("feed_sano.csv")
print("Feed sano:", validar_feed(df_sano))

df_corrupto = pd.read_csv("feed_corrupto.csv")
print("Feed corrupto:")
for e in validar_feed(df_corrupto):
    print("  -", e)

df_corrupto2 = df_corrupto.rename(columns={"city": "ciudad"})
for e in validar_feed(df_corrupto2):
    print("  -", e)

Feed sano: []
Feed corrupto:
  - Esquema cambió. Faltan: {'ciudad'} | Sobran: {'city'}
  - 40 precios no son numéricos
  - 15 nulos en calificacion
  - 8 calificaciones fuera de rango [3-5]


In [13]:

%pip install pandera

  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.1 MB 1.2 MB/s eta 0:00:02
   --------------- ------------------------ 0.8/2.1 MB 1.2 MB/s eta 0:00:02
   -------------------- ------------------- 1.0/2.1 MB 1.2 MB/s eta 0:00:01
   ------------------------- -------------- 1.3/2.1 MB 1.2 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.1 MB 1.2 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.1 MB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 1.2 MB/s  0:00:01
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 k

In [5]:
import pandera.pandas as pa
from pandera.pandas import Column, DataFrameSchema, Check

esquema = DataFrameSchema({
    "pedido_id":      Column(str),
    "restaurante_id": Column(int),
    "categoria":      Column(str),
    "ciudad":         Column(str),
    "precio":         Column(float),
    "calificacion":   Column(float, Check.in_range(3.0, 5.0), nullable=False),
})

In [6]:
df_corrupto = pd.read_csv("feed_corrupto.csv")

try:
    esquema.validate(df_corrupto, lazy=True)
    print("Feed válido")
except pa.errors.SchemaErrors as e:
    resumen = e.failure_cases.groupby(["column", "check"]).size()
    print(resumen)

column        check             
calificacion  in_range(3.0, 5.0)     8
              not_nullable          15
precio        dtype('float64')      40
dtype: int64


In [7]:
df_sano = pd.read_csv("feed_sano.csv")
print(esquema.validate(df_sano, lazy=True).shape)

(200, 6)


In [8]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("pipeline_rappi")

def enviar_alerta(mensaje):
    logger.error(f"ALERTA: {mensaje}")

In [9]:
import pandas as pd
import logging
from sqlalchemy import create_engine, text

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("pipeline_rappi")

ESQUEMA = {"pedido_id", "restaurante_id", "categoria",
           "ciudad", "precio", "calificacion"}

def enviar_alerta(mensaje):
    logger.error(f"ALERTA: {mensaje}")

def validar(df):
    errores = []
    if set(df.columns) != ESQUEMA:
        errores.append(f"Esquema cambió. Recibido: {set(df.columns)}")
        return errores
    if pd.to_numeric(df["precio"], errors="coerce").isna().any():
        errores.append("Hay precios no numéricos")
    if df["calificacion"].isna().any():
        errores.append(f"{int(df['calificacion'].isna().sum())} nulos en calificacion")
    return errores

def procesar_feed(ruta):
    try:
        df = pd.read_csv(ruta)
    except FileNotFoundError:
        enviar_alerta(f"No se encontró el archivo {ruta}")
        return None

    errores = validar(df)
    if errores:
        for e in errores:
            enviar_alerta(e)
        logger.warning(f"Feed {ruta} RECHAZADO. No se persiste.")
        return None

    logger.info(f"Feed {ruta} VÁLIDO. Persistiendo {len(df)} filas...")
    engine = create_engine("sqlite:///rappi_calidad.db")
    df.to_sql("pedidos_validados", engine, if_exists="replace", index=False)

    with engine.connect() as conn:
        total = conn.execute(
            text("SELECT COUNT(*) FROM pedidos_validados")).scalar()
        nulos = conn.execute(
            text("SELECT COUNT(*) FROM pedidos_validados "
                 "WHERE calificacion IS NULL")).scalar()
    logger.info(f"Verificación SQL → filas: {total}, nulos: {nulos}")
    return df

In [10]:
print("INTENTO 1 — feed corrupto (debe rechazarse):")
procesar_feed("feed_corrupto.csv")

print("\nINTENTO 2 — feed sano (debe persistirse):")
procesar_feed("feed_sano.csv")

2026-07-05 19:25:20,178 | ERROR | ALERTA: Esquema cambió. Recibido: {'pedido_id', 'restaurante_id', 'precio', 'calificacion', 'categoria', 'city'}
2026-07-05 19:25:20,180 | WARNING | Feed feed_corrupto.csv RECHAZADO. No se persiste.
2026-07-05 19:25:20,186 | INFO | Feed feed_sano.csv VÁLIDO. Persistiendo 200 filas...
2026-07-05 19:25:20,282 | INFO | Verificación SQL → filas: 200, nulos: 0


INTENTO 1 — feed corrupto (debe rechazarse):

INTENTO 2 — feed sano (debe persistirse):


,pedido_id,restaurante_id,categoria,ciudad,precio,calificacion
0,PED1000,39,Ensaladas,Medellín,57430.0,4.0
1,PED1001,29,Sushi,Medellín,44275.0,3.8
2,PED1002,15,Bebidas,Cali,53654.0,4.6
3,PED1003,43,Pasta,Barranquilla,46599.0,4.3
4,PED1004,8,Burgers,Medellín,40682.0,4.6
...,...,...,...,...,...,...
195,PED1195,37,Sushi,Barranquilla,54534.0,3.7
196,PED1196,33,Pizza,Barranquilla,63175.0,4.5
197,PED1197,42,Pasta,Cali,13039.0,4.4
198,PED1198,44,Pizza,Medellín,53134.0,3.3
